In [0]:
# ============================================================
# CELL 1: Read from Silver Layer
# ============================================================

spark.sql("USE healthcare_db")

df_silver = spark.table("healthcare_db.silver_patients")

print(f"✅ Loaded Silver table: {df_silver.count()} rows")
display(df_silver.limit(3))

✅ Loaded Silver table: 1000 rows


patient_id,name,age,gender,diagnosis,admission_date,discharge_date,hospital,treatment_cost,readmitted,length_of_stay,age_group,cost_category
P0001,Patient_1,21,OTHER,HEART FAILURE,2023-10-17,2023-10-21,Green Valley Medical,11548.93,YES,4,18-29,Medium
P0002,Patient_2,87,MALE,KIDNEY DISEASE,2023-11-24,2023-12-18,City General Hospital,1974.96,YES,24,75+,Low
P0003,Patient_3,21,OTHER,COVID-19,2022-08-27,2022-09-13,Lakeside Clinic,21266.23,NO,17,18-29,High


In [0]:
# ============================================================
# CELL 2: Gold Table 1 — Hospital Performance Summary
# ============================================================

# This table answers: Which hospitals have the highest costs,
# longest stays, and worst readmission rates?
# Hospital admins and operations teams use exactly this kind of report.

from pyspark.sql.functions import (
    count, round, avg, sum as spark_sum,
    when, col
)

gold_hospital = df_silver.groupBy("hospital").agg(

    count("patient_id")
        .alias("total_patients"),

    round(avg("length_of_stay"), 2)
        .alias("avg_length_of_stay"),

    round(avg("treatment_cost"), 2)
        .alias("avg_treatment_cost"),

    round(spark_sum("treatment_cost"), 2)
        .alias("total_revenue"),

    # Readmission rate = readmitted patients / total patients * 100
    round(
        (spark_sum(when(col("readmitted") == "YES", 1).otherwise(0)) /
         count("patient_id")) * 100, 2
    ).alias("readmission_rate_pct")

).orderBy("total_patients", ascending=False)

gold_hospital.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare_db.gold_hospital_summary")

print("✅ Gold Table 1 saved: gold_hospital_summary")
display(gold_hospital)

✅ Gold Table 1 saved: gold_hospital_summary


hospital,total_patients,avg_length_of_stay,avg_treatment_cost,total_revenue,readmission_rate_pct
Sunrise Health Center,210,16.37,26257.63,5514102.13,48.1
Metro Care Hospital,204,16.31,25254.88,5151995.59,51.96
Lakeside Clinic,196,15.87,25920.32,5080382.21,52.04
Green Valley Medical,195,16.29,24112.95,4702025.45,39.49
City General Hospital,195,15.32,23858.45,4652398.56,41.54


In [0]:
# ============================================================
# CELL 3: Gold Table 2 — Diagnosis Summary
# ============================================================

# Which diagnoses are most common, most expensive, 
# and have the longest treatment durations?
# Useful for resource planning and insurance analysis.

gold_diagnosis = df_silver.groupBy("diagnosis").agg(

    count("patient_id")
        .alias("total_patients"),

    round(avg("treatment_cost"), 2)
        .alias("avg_cost"),

    round(spark_sum("treatment_cost"), 2)
        .alias("total_cost"),

    round(avg("length_of_stay"), 2)
        .alias("avg_length_of_stay"),

    round(
        (spark_sum(when(col("readmitted") == "YES", 1).otherwise(0)) /
         count("patient_id")) * 100, 2
    ).alias("readmission_rate_pct")

).orderBy("total_cost", ascending=False)

gold_diagnosis.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare_db.gold_diagnosis_summary")

print("✅ Gold Table 2 saved: gold_diagnosis_summary")
display(gold_diagnosis)

✅ Gold Table 2 saved: gold_diagnosis_summary


diagnosis,total_patients,avg_cost,total_cost,avg_length_of_stay,readmission_rate_pct
HYPERTENSION,145,24368.72,3533464.65,15.94,43.45
CANCER,143,24582.82,3515342.84,16.25,48.95
ASTHMA,124,25919.78,3214052.24,16.61,45.97
HEART FAILURE,121,26190.38,3169036.5,15.9,42.98
KIDNEY DISEASE,123,25350.8,3118147.92,16.16,46.34
PNEUMONIA,115,25716.57,2957406.03,15.93,53.91
DIABETES,112,25942.33,2905540.9,15.99,44.64
COVID-19,117,22973.61,2687912.86,15.47,47.86


In [0]:
# ============================================================
# CELL 4: Gold Table 3 — Age Group Summary
# ============================================================

# Which age groups are most vulnerable and most expensive to treat?
# Critical for public health policy and hospital resource allocation.

gold_age = df_silver.groupBy("age_group", "gender").agg(

    count("patient_id")
        .alias("total_patients"),

    round(avg("treatment_cost"), 2)
        .alias("avg_cost"),

    round(avg("length_of_stay"), 2)
        .alias("avg_length_of_stay"),

    round(
        (spark_sum(when(col("readmitted") == "YES", 1).otherwise(0)) /
         count("patient_id")) * 100, 2
    ).alias("readmission_rate_pct")

).orderBy("age_group", "gender")

gold_age.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare_db.gold_age_summary")

print("✅ Gold Table 3 saved: gold_age_summary")
display(gold_age)

✅ Gold Table 3 saved: gold_age_summary


age_group,gender,total_patients,avg_cost,avg_length_of_stay,readmission_rate_pct
18-29,FEMALE,47,22622.32,12.43,40.43
18-29,MALE,51,30118.41,14.69,50.98
18-29,OTHER,48,22238.68,15.79,43.75
30-44,FEMALE,67,27672.27,17.55,53.73
30-44,MALE,64,24965.12,16.41,46.88
30-44,OTHER,77,24977.04,15.95,46.75
45-59,FEMALE,61,26107.99,14.36,40.98
45-59,MALE,77,26031.92,17.92,38.96
45-59,OTHER,72,24195.27,16.08,48.61
60-74,FEMALE,75,25957.11,15.83,48.0


In [0]:
# ============================================================
# CELL 5: Gold Table 4 — Monthly Admissions Trend
# ============================================================

# How many patients were admitted each month?
# Are costs rising over time? Are there seasonal patterns?
# This is a TIME SERIES analysis — very common in healthcare reporting.

from pyspark.sql.functions import date_format, year, month

gold_monthly = df_silver \
    .withColumn("year",       year(col("admission_date"))) \
    .withColumn("month",      month(col("admission_date"))) \
    .withColumn("year_month", date_format(col("admission_date"), "yyyy-MM")) \
    .groupBy("year", "month", "year_month").agg(

        count("patient_id")
            .alias("total_admissions"),

        round(avg("treatment_cost"), 2)
            .alias("avg_cost"),

        round(avg("length_of_stay"), 2)
            .alias("avg_length_of_stay")

    ).orderBy("year", "month")

gold_monthly.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare_db.gold_monthly_admissions")

print("✅ Gold Table 4 saved: gold_monthly_admissions")
display(gold_monthly)

✅ Gold Table 4 saved: gold_monthly_admissions


year,month,year_month,total_admissions,avg_cost,avg_length_of_stay
2022,1,2022-01,32,30785.02,13.69
2022,2,2022-02,39,23226.11,14.9
2022,3,2022-03,50,22268.17,16.64
2022,4,2022-04,50,27612.1,18.02
2022,5,2022-05,43,26683.61,17.23
2022,6,2022-06,49,22426.82,14.86
2022,7,2022-07,35,23684.61,13.97
2022,8,2022-08,47,24107.22,16.11
2022,9,2022-09,40,27502.27,15.95
2022,10,2022-10,37,25998.97,14.38


In [0]:
# ============================================================
# CELL 6: Verify All Gold Tables Exist
# ============================================================

spark.sql("USE healthcare_db")

tables = spark.sql("SHOW TABLES IN healthcare_db")
display(tables)

database,tableName,isTemporary
healthcare_db,bronze_patients,false
healthcare_db,gold_age_summary,false
healthcare_db,gold_diagnosis_summary,false
healthcare_db,gold_hospital_summary,false
healthcare_db,gold_monthly_admissions,false
healthcare_db,silver_patients,false


In [0]:
# ============================================================
# CELL 7: Final Cross-Table SQL — Top Insight
# ============================================================

# Which diagnosis has the highest readmission rate 
# AND the highest average cost? (The most problematic ones)
# This kind of query directly informs hospital management decisions.

result = spark.sql("""
    SELECT
        diagnosis,
        total_patients,
        avg_cost,
        avg_length_of_stay,
        readmission_rate_pct,
        RANK() OVER (ORDER BY readmission_rate_pct DESC) AS readmission_rank,
        RANK() OVER (ORDER BY avg_cost DESC)             AS cost_rank
    FROM healthcare_db.gold_diagnosis_summary
    ORDER BY readmission_rate_pct DESC
""")

display(result)

diagnosis,total_patients,avg_cost,avg_length_of_stay,readmission_rate_pct,readmission_rank,cost_rank
PNEUMONIA,115,25716.57,15.93,53.91,1,4
CANCER,143,24582.82,16.25,48.95,2,6
COVID-19,117,22973.61,15.47,47.86,3,8
KIDNEY DISEASE,123,25350.8,16.16,46.34,4,5
ASTHMA,124,25919.78,16.61,45.97,5,3
DIABETES,112,25942.33,15.99,44.64,6,2
HYPERTENSION,145,24368.72,15.94,43.45,7,7
HEART FAILURE,121,26190.38,15.9,42.98,8,1
